### https://www.kaggle.com/competitions/drawing-with-llms

In [30]:
!pip install -q -U google-genai

In [31]:
import kagglehub
import pandas as pd
import re

#train_path = kagglehub.competition_download('drawing-with-llms', 'train.csv')
df = pd.read_csv('./drawing-with-llms/train_master_description_data.csv')
print(df.shape)
df.head(2)

(4188, 1)


,description
0,'Golden wheat fields under a setting sun'
1,'Vibrant orange circles on a cobalt blue backg...


In [32]:
import openai
import pandas as pd
import time
import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
gemini_api_key=os.getenv("GEMINI_API_KEY")

In [33]:
from openai import OpenAI
client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello are you"},
    ],
    stream=False
)

print(response.choices[0].message.content)

Hello! Yes, I'm here and ready to help. How can I assist you today? 😊


In [34]:
# Function to get summary from OpenAI API
def get_svg_code(text):
    instruction = f"""
            Generate SVG code to visually represent the following text description, while respecting the given constraints.
            <constraints>
            * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
            * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
            </constraints>

            Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints. 
            Focus on a clear and concise representation of the input description within the given limitations. 
            Always give the complete SVG code with nothing omitted. Never use an ellipsis.

            The code is scored based on similarity to the description, Visual question anwering and aesthetic components.
            Please generate a svg code considering similarity, visual question answering and aesthetic compoenents.

            input description: {text}
            """

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "Generate a SVG code as per instruction"},
                {"role": "user", "content": instruction}
            ],
            temperature=0.7,
            max_tokens=3200
        )
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"Error: {e}")
        return None

In [35]:
# from tqdm import tqdm
# tqdm.pandas()
# df["deepseek_response"] = df.progress_apply(lambda row: get_svg_code(row["description"]), axis=1)

In [36]:
from google import genai
client = genai.Client(api_key=gemini_api_key)

# Function to get summary from OpenAI API
def get_svg_code_gemini(text):
    instruction = f"""
            Generate SVG code to visually represent the following text description, while respecting the given constraints.
            <constraints>
            * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
            * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
            </constraints>

            Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints. 
            Focus on a clear and concise representation of the input description within the given limitations. 
            Always give the complete SVG code with nothing omitted. Never use an ellipsis.

            The code is scored based on similarity to the description, Visual question anwering and aesthetic components.
            Please generate a detailed svg code considering similarity, visual question answering and aesthetic compoenents.
            
            input description: {text}
            """

    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash", contents= instruction
        )
        return response
    
    except Exception as e:
        print(f"Error: {e}")
        return None



In [38]:
import pandas as pd
from tqdm import tqdm
import os

tqdm.pandas()

batch_size = 100
total_rows = len(df)

# Optional: directory to store batches
os.makedirs("batches", exist_ok=True)

for i in range(0, total_rows, batch_size):
    batch_num = i // batch_size + 1
    filename = f"batches/response_batch_{batch_num}.csv"
    
    if os.path.exists(filename):
        print(f"Skipping batch {batch_num}, already exists.")
        continue

    batch_df = df.iloc[i:i+batch_size].copy()
    
    # Apply your function here
    batch_df["gemini_response"] = batch_df.progress_apply(
        lambda row: get_svg_code_gemini(row["description"]), axis=1
    )

    # Save after processing
    batch_df.to_csv(filename, index=False)
    print(f"Saved batch {batch_num} to {filename}")


100%|█████████████████████████████████████████| 100/100 [07:00<00:00,  4.20s/it]


Saved batch 1 to batches/response_batch_1.csv


100%|█████████████████████████████████████████| 100/100 [06:58<00:00,  4.18s/it]


Saved batch 2 to batches/response_batch_2.csv


100%|█████████████████████████████████████████| 100/100 [07:04<00:00,  4.24s/it]


...

Saved batch 40 to batches/response_batch_40.csv


100%|█████████████████████████████████████████| 100/100 [05:21<00:00,  3.22s/it]


Saved batch 41 to batches/response_batch_41.csv


100%|███████████████████████████████████████████| 88/88 [04:37<00:00,  3.15s/it]

Saved batch 42 to batches/response_batch_42.csv
